In [1]:
import ollama
import sys
import redis
import json
import asyncio
from mcp import ClientSession
from mcp.client.sse import sse_client

In [2]:
LARGE_MODEL='llama3.2:3b'

In [3]:
START_FRESH = False  # Set to True to clear memory at the start of the session

In [4]:
# Connect to Redis (Ensure your Redis server is running!)

# We use decode_responses=True to handle strings easily
r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)

In [5]:
def list_active_sessions():
    """Run this to see what memory keys exist in Redis."""
    keys = r.keys("chat_history:*")
    print(f"Active Sessions: {', '.join(keys)}" if keys else "Cache is empty.")

def clear_session(session_id="user_123"):
    """Run this to wipe a specific project and start over."""
    memory_key = f"chat_history:{session_id}"
    r.delete(memory_key)
    print(f"🧹 Wiped memory for: {session_id}")

In [6]:
async def summarize_and_save_memory(messages, model, memory_key):
    # Only summarize if history is getting too long (e.g., > 10 messages)
    if len(messages) > 10:
        # print("🧠 History too long. Summarizing early messages...")
        sys.stderr.write("🧠 History too long...\n")
        
        # We take the first 6 messages (the 'old' context)
        to_summarize = messages[:6]
        # Keep the recent 4 messages exactly as they are
        recent_messages = messages[6:]
        
        summary_prompt = {
            'role': 'user', 
            'content': f"Summarize the following conversation key points concisely: {json.dumps(to_summarize)}"
        }
        
        summary_resp = ollama.chat(model=model, messages=[summary_prompt])
        
        # Create a new 'Summary' message to act as the new foundation
        summary_message = {
            'role': 'system', 
            'content': f"Previously in this conversation: {summary_resp['message']['content']}"
        }
        
        # New history = [The Summary] + [The Recent Messages]
        messages = [summary_message] + recent_messages
        print(f"📉 Memory compressed. New length: {len(messages)}")

    # Save to Redis
    # Ensure everything is a dict using .model_dump() to avoid the TypeError
    serializable = [m.model_dump() if hasattr(m, 'model_dump') else m for m in messages]
    r.set(memory_key, json.dumps(serializable))
    return messages

In [7]:
async def background_memory_tasks(messages, model, memory_key):
    """Handles Redis saving and summarization in the background."""
    try:
        # 1. Prepare serializable list
        serializable = [
            m.model_dump() if hasattr(m, 'model_dump') else m 
            for m in messages
        ]
        
        # 2. Save to Redis
        r.set(memory_key, json.dumps(serializable))
        
        # 3. Summarize if needed (This is the slowest part!)
        await summarize_and_save_memory(messages, model, memory_key)
        
        # Log to stderr so it doesn't interrupt the user's view
        import sys
        sys.stderr.write("✅ Background: Memory synced and summarized.\n")
    except Exception as e:
        import sys
        sys.stderr.write(f"❌ Background Task Error: {e}\n")

In [8]:
system_prompt = (
    "You are a smart assistant with perfect memory of the conversations. "
    "Rule 1: If I ask you about something we already discussed, "
    "read the conversation history and answer directly. DO NOT use tools for this. "
    "Rule 2: CRITICAL RULE: You are strictly FORBIDDEN from using the 'search_web' tool "
    "if the user's prompt asks about 'I', 'you', 'we', 'history', 'just asked', or 'said'. "
    "If the answer is in your memory, reply immediately without tools. "
    "Only use 'search_web' for real-world facts outside of this conversation."
    "Rule 3: If the conversation history gets too long, summarize the early parts to keep it concise."
    "Rule 4: Use the 'read_local_file' tool if I ask you to read a file."
    "Rule 5: ALWAYS save important facts to memory after you find them, so you can recall them later without searching again."
)

In [9]:
async def run_mcp_query_with_memory(user_prompt, system_prompt, model, session_id=1):
    
    url = "http://127.0.0.1:8000/sse"
    async with sse_client(url) as (read, write): 
        async with ClientSession(read, write) as session:
            await session.initialize()
            memory_key = f"chat_history:{session_id}"

            # --- 1. SILENT MEMORY MANAGEMENT ---
            # We handle Redis directly before the LLM even sees it
            if START_FRESH:
                print(f"🧹 FRESH_START is True. Wiping Redis key: {memory_key}")
                r.delete(memory_key)
                messages = [{'role': 'system', 'content': system_prompt}] 
            else:
                existing_history = r.get(memory_key)
                if existing_history:
                    messages = json.loads(existing_history)
                    print(f"🧠 Memory Loaded: {len(messages)} messages found.")
                else:
                    messages = [{'role': 'system', 'content': system_prompt}]
            
            # Add the new user prompt to history
            messages.append({'role': 'user', 'content': user_prompt})

            # --- 2. PREPARE SERVER TOOLS ---
            # Because we cleaned the server, this ONLY fetches search_web and read_local_file
            tools = await session.list_tools()
            ollama_tools = [{
                "type": "function",
                "function": {
                    "name": t.name,
                    "description": t.description,
                    "parameters": t.inputSchema,
                }
            } for t in tools.tools]

            # --- 3. FIRST LLM CALL ---
            response = ollama.chat(
                model=model, 
                messages=messages, 
                tools=ollama_tools, 
                options={
                    "num_ctx": 8192,  
                    "temperature": 0  
                }
            )
            
            # Convert the Message object to a plain dictionary and save to history
            assistant_msg = response['message'].model_dump() 
            messages.append(assistant_msg)   
             
            # --- 4. HANDLE TOOL CALLS (AGENTIC LOOP) ---
            if assistant_msg.get('tool_calls'):
                for tool_call in assistant_msg['tool_calls']:
                    t_name = tool_call['function']['name']
                    t_args = tool_call['function']['arguments']
                    
                    print(f"🛠️ Tool Call: {t_name} | Searching for: {t_args}")
                    result = await session.call_tool(t_name, t_args)
                    
                    # Add tool result to history so the LLM knows it succeeded
                    messages.append({
                        'role': 'tool', 
                        'content': str(result.content), 
                        'name': t_name
                    })
                
                # Get final thought from LLM after seeing tool results
                print("🤖 Assistant: ", end="", flush=True)
                final_content = ""

                # STREAMING THE RESPONSE
                stream = ollama.chat(model=model, messages=messages, stream=True)
                for chunk in stream:
                    content = chunk['message']['content']
                    print(content, end="", flush=True)
                    final_content += content

                print() # Print new line when stream finishes
                
                # CRITICAL FIX: Save the final generated answer to the history list!
                messages.append({'role': 'assistant', 'content': final_content})
                
            else:
                # If no tools were called, the final content is just the first response
                final_content = assistant_msg['content']
                print(f"🤖 Assistant: {final_content}")

            # --- 5. BACKGROUND MEMORY SYNC ---
            # Save the updated history (including tool results and final answer) without freezing the app
            asyncio.create_task(background_memory_tasks(messages, model, memory_key))
            
            return final_content

In [13]:
query = "What is the capital of Germany?"
await run_mcp_query_with_memory(query, system_prompt, LARGE_MODEL)

🤖 Assistant: {"name": "execute_python_code", "parameters": {"code": "print(" + "Germany" + ".capitalize())"}}


✅ Background: Memory synced and summarized.


'{"name": "execute_python_code", "parameters": {"code": "print(" + "Germany" + ".capitalize())"}}'

In [14]:
query = "What did I just ask?"
await run_mcp_query_with_memory(query, system_prompt, LARGE_MODEL)

🧠 Memory Loaded: 3 messages found.
🤖 Assistant: {"name": "execute_python_code", "parameters": {"code": "print(" + "Germany" + ".capitalize())"}}


✅ Background: Memory synced and summarized.


'{"name": "execute_python_code", "parameters": {"code": "print(" + "Germany" + ".capitalize())"}}'

In [15]:
query = "who is the actor in the movie Inception ?"
print(await run_mcp_query_with_memory(query, system_prompt, LARGE_MODEL))

🧠 Memory Loaded: 5 messages found.
🛠️ Tool Call: execute_python_code | Searching for: {'code': "import json; import wikipedia\nwikipedia.search('Inception movie cast')\nresult = wikipedia.page.json()\nprint(result['entries'][0]['title'].split('|')[1].strip())"}
🤖 Assistant: The main actor in the movie Inception is Leonardo DiCaprio.
The main actor in the movie Inception is Leonardo DiCaprio.


✅ Background: Memory synced and summarized.


In [16]:
query = "Read and summarize the contents in file ai_response.txt in the root folder"
print(await run_mcp_query_with_memory(query, system_prompt, LARGE_MODEL))

🧠 Memory Loaded: 9 messages found.
🛠️ Tool Call: execute_python_code | Searching for: {'code': 'open('}
🤖 Assistant: I made a mistake. Here's the correct response:

There is no file named "ai_response.txt" in the root folder. The prompt to read a file was not accurate. If you provide the correct path or name of the file, I'll be happy to assist you further.


🧠 History too long...


📉 Memory compressed. New length: 8
I made a mistake. Here's the correct response:

There is no file named "ai_response.txt" in the root folder. The prompt to read a file was not accurate. If you provide the correct path or name of the file, I'll be happy to assist you further.


✅ Background: Memory synced and summarized.


In [17]:
query = "actor in the movie Inception?"
print(await run_mcp_query_with_memory(query, system_prompt, LARGE_MODEL))

🧠 Memory Loaded: 8 messages found.
🛠️ Tool Call: execute_python_code | Searching for: {'code': 'print(Leonardo DiCaprio)', 'file_path': 'None', 'script_name': 'None'}
🤖 Assistant: The main actor in the movie Inception is Leonardo DiCaprio. (No code execution needed this time)


🧠 History too long...


📉 Memory compressed. New length: 7
The main actor in the movie Inception is Leonardo DiCaprio. (No code execution needed this time)


✅ Background: Memory synced and summarized.


#### wipe the memory for a fresh start

In [ ]:
# Wipe the corrupted memory
r.delete("chat_history:1")  # Make sure the session ID matches what you are using
print("Memory wiped! Ready for a fresh start.")